# Lab 5.1 — An electron as a probability cloud

**Goal:** sample thousands of positions of hydrogen's electron and see the orbital appear as a cloud, then check where the electron is *most likely* to be found.

For the 1s state the probability of finding the electron at distance $r$ (in any direction) is

$$P(r) = \frac{4 r^2}{a_0^3}\, e^{-2r/a_0}$$

which peaks exactly at the Bohr radius $a_0 = 0.0529$ nm. Bohr's orbit survives as the *most likely* distance.

In [ ]:
import sys; sys.path.append("..")
from constants import a_0
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(seed=1)

## 1. Radial probability

In [ ]:
r = np.linspace(0, 6, 600)          # in units of a_0
P = 4 * r**2 * np.exp(-2 * r)       # P(r) with a_0 = 1

plt.figure(figsize=(7, 4))
plt.plot(r, P)
plt.axvline(1, ls="--", color="gray", label="Bohr radius a₀")
plt.xlabel("r / a₀"); plt.ylabel("P(r)")
plt.title("Where is hydrogen's 1s electron?")
plt.legend(); plt.show()
print(f"Most likely r = {r[np.argmax(P)]:.3f} a0 = {r[np.argmax(P)] * a_0 * 1e9:.4f} nm")
print(f"Average r     = {np.trapezoid(r * P, r):.3f} a0   (exact answer: 1.5 a0)")

## 2. Sample the cloud

$P(r)$ is a Gamma distribution with shape 3 and scale $a_0/2$, so NumPy can sample it directly. Directions are uniform on a sphere.

In [ ]:
N = 20_000
r_s = rng.gamma(shape=3, scale=0.5, size=N)      # distances in a_0
cos_t = rng.uniform(-1, 1, N)                    # uniform directions
phi = rng.uniform(0, 2 * np.pi, N)
sin_t = np.sqrt(1 - cos_t**2)
x, y, z = r_s * sin_t * np.cos(phi), r_s * sin_t * np.sin(phi), r_s * cos_t

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(x, y, z, s=0.5, alpha=0.25)
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_zlim(-4, 4)
ax.set_title("1s orbital: 20,000 sampled positions (units of a₀)")
plt.show()

## 3. Check the sample against theory

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(r_s, bins=80, density=True, alpha=0.6, label="sampled")
plt.plot(r, P, lw=2, label="theory P(r)")
plt.xlabel("r / a₀"); plt.legend(); plt.show()
print(f"Sample mean r = {r_s.mean():.3f} a0 (theory 1.5)")
print(f"Chance the electron is farther than 3 a0: {np.mean(r_s > 3):.3%}")

## Newton vs quantum

Bohr's (Newtonian) orbit puts the electron at exactly $r = a_0$ every time. The quantum cloud says $a_0$ is only the *most likely* distance, and the average is $1.5\,a_0$.

## Exercises

1. **[Core]** What fraction of samples lie inside $r < a_0$?
2. **[Extension]** Rotate the 3D plot. Why does the 1s cloud look the same from every angle?
3. **[Challenge]** For the 2p$_z$ orbital, $|\psi|^2 \propto r^2 e^{-r/a_0}\cos^2\theta$. Sample it with rejection sampling and plot the dumbbell. Compare with the 3D visual in `visuals/05-orbitals`.